# Question 3 — Real-Time Weather Streaming for Seoul

## Why this API ?

In order to obtain more details about the weather in Seoul, using the Open-Meteo API in conjunction with Spark Streaming, it is possible to demonstrate how a streaming pipeline makes this analysis even more complete. Temperature was identified as a key factor in understanding bike rental demand, highlighting the need for real-time weather monitoring in Seoul over time, rather than a single snapshot. The use of various climate metrics helps anticipate demand peaks, enabling better planning and preparation.

* The Open-Meteo Weather API was selected as the online data source. The API provides real-time weather information including temperature and wind speed for Seoul.

In [1]:
# Import libraries
import requests                                                            # HTTP requests to the API
import time                                                                # delays between API calls
from datetime import datetime                                              # timestamps
from pyspark.sql import SparkSession                                       # Spark entry point
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType  # Spark data types
from pyspark.sql.functions import avg, max, min, round, hour, stddev,col, when  # aggregation functions

# Create SparkSession
spark = SparkSession.builder \
    .appName("Seoul_Weather_Streaming") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/07/02 21:51:34 WARN Utils: Your hostname, DESKTOP-DUGUN4Q resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/02 21:51:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 21:51:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/02 21:51:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 33848)
Traceback (most recent call last):
  File "/usr/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request

## Define the Data Schema

In [2]:
url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=37.5665&longitude=126.9780"
    "&current=temperature_2m,apparent_temperature,wind_speed_10m,relative_humidity_2m"
    ",precipitation,rain,snowfall,cloud_cover,wind_gusts_10m"
)

schema = StructType([
    StructField("city",                 StringType(),    True),
    StructField("temperature",          DoubleType(),    True),
    StructField("apparent_temperature", DoubleType(),    True),
    StructField("wind_speed",           DoubleType(),    True),
    StructField("humidity",             DoubleType(),    True),
    StructField("precipitation",        DoubleType(),    True),
    StructField("rain",                 DoubleType(),    True),
    StructField("snowfall",             DoubleType(),    True),
    StructField("cloud_cover",          DoubleType(),    True),
    StructField("wind_gusts",           DoubleType(),    True), 
    StructField("timestamp",            TimestampType(), True),
])

## Spark Streaming to Process Incoming Data

The application continuously retrieves weather data from the Open-Meteo API at 15-minute intervals, collecting 30 records over an approximately 8-hour window to simulate a real-time streaming source. This interval was chosen to balance meaningful data variation with a reasonable execution time. Each record is collected with a timestamp and stored for processing.

In [3]:
# Polling loop — collects 30 records from the API, one every 15 minutes
records = []
total   = 30

for i in range(total):
    try:
        response = requests.get(url, timeout=10)         # fetch data from API
        data     = response.json()
        current  = data["current"]

        record = (
            "Seoul",
            float(current["temperature_2m"]),
            float(current["apparent_temperature"]),
            float(current["wind_speed_10m"]),
            float(current["relative_humidity_2m"]),
            float(current["precipitation"]),
            float(current["rain"]),
            float(current["snowfall"]),
            float(current["cloud_cover"]),
            float(current["wind_gusts_10m"]),
            datetime.now(),
        )
        records.append(record)
        print(f"Record {i+1}/{total} → {record}")

    except Exception as e:
        print(f"Error on record {i+1}: {e}")

    time.sleep(900)                                        # wait 900 second between calls

Record 1/30 → ('Seoul', 20.7, 24.5, 2.6, 99.0, 0.0, 0.0, 0.0, 80.0, 9.7, datetime.datetime(2026, 7, 2, 21, 51, 49, 134276))
Record 2/30 → ('Seoul', 20.8, 24.6, 2.6, 99.0, 0.0, 0.0, 0.0, 88.0, 10.4, datetime.datetime(2026, 7, 2, 22, 7, 28, 688151))
Record 3/30 → ('Seoul', 20.8, 24.7, 2.6, 99.0, 0.0, 0.0, 0.0, 93.0, 11.2, datetime.datetime(2026, 7, 2, 22, 23, 8, 868452))
Record 4/30 → ('Seoul', 20.8, 24.7, 2.4, 99.0, 0.0, 0.0, 0.0, 95.0, 12.2, datetime.datetime(2026, 7, 2, 22, 38, 49, 170759))
Record 5/30 → ('Seoul', 20.9, 24.8, 2.3, 99.0, 0.0, 0.0, 0.0, 97.0, 14.4, datetime.datetime(2026, 7, 2, 22, 54, 27, 879514))
Record 6/30 → ('Seoul', 20.9, 24.9, 2.3, 99.0, 0.0, 0.0, 0.0, 98.0, 15.8, datetime.datetime(2026, 7, 2, 23, 10, 8, 157380))
Record 7/30 → ('Seoul', 21.0, 24.9, 2.3, 99.0, 0.0, 0.0, 0.0, 99.0, 16.6, datetime.datetime(2026, 7, 2, 23, 25, 47, 124913))
Record 8/30 → ('Seoul', 21.0, 25.0, 2.5, 99.0, 0.0, 0.0, 0.0, 100.0, 17.3, datetime.datetime(2026, 7, 2, 23, 41, 25, 452367))
Rec

## Data Ingestion and Processing

In [4]:
# Convert collected records into Spark DataFrame
df = spark.createDataFrame(records, schema = schema)

# Show collected data
print("Weather Data:")
df.printSchema()  
df.show(10, truncate = False)

Weather Data:
root
 |-- city: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- cloud_cover: double (nullable = true)
 |-- wind_gusts: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



+-----+-----------+--------------------+----------+--------+-------------+----+--------+-----------+----------+--------------------------+
|city |temperature|apparent_temperature|wind_speed|humidity|precipitation|rain|snowfall|cloud_cover|wind_gusts|timestamp                 |
+-----+-----------+--------------------+----------+--------+-------------+----+--------+-----------+----------+--------------------------+
|Seoul|20.7       |24.5                |2.6       |99.0    |0.0          |0.0 |0.0     |80.0       |9.7       |2026-07-02 21:51:49.134276|
|Seoul|20.8       |24.6                |2.6       |99.0    |0.0          |0.0 |0.0     |88.0       |10.4      |2026-07-02 22:07:28.688151|
|Seoul|20.8       |24.7                |2.6       |99.0    |0.0          |0.0 |0.0     |93.0       |11.2      |2026-07-02 22:23:08.868452|
|Seoul|20.8       |24.7                |2.4       |99.0    |0.0          |0.0 |0.0     |95.0       |12.2      |2026-07-02 22:38:49.170759|
|Seoul|20.9       |24.8    

In [5]:
# Check the count of records
print("Total records collected:", df.count())

Total records collected: 30


In [14]:
# Save collected data to CSV 
(df.coalesce(1)                                    # merge into a single file
   .write
   .mode("overwrite")                              # overwrite if file already exists
   .option("header", True)                         # include column headers
   .csv("weather_seoul_streaming.csv"))            # save as CSV

print("Data saved successfully.")

Data saved successfully.


## Perform Real-Time Analysis (Aggregation)

In [6]:
# Temperature

print("Temperature Summary Analysis:")                 # Display analysis heading

df.select(
    avg("temperature").alias("average_temperature"),   # Calculate average temperature
    max("temperature").alias("highest_temperature"),   # Calculate highest temperature
    min("temperature").alias("lowest_temperature")     # Calculate lowest temperature
).show()    

Temperature Summary Analysis:
+-------------------+-------------------+------------------+
|average_temperature|highest_temperature|lowest_temperature|
+-------------------+-------------------+------------------+
| 23.209999999999994|               27.0|              20.7|
+-------------------+-------------------+------------------+



In [7]:
# Wind Speed

print("Wind Summary Analysis:")                         # Display analysis heading

df.select(
    avg("wind_speed").alias("average_windspeed"),       # Calculate average wind speed
    max("wind_speed").alias("highest_windspeed"),       # Calculate highest wind speed
    min("wind_speed").alias("lowest_windspeed")         # Calculate lowest wind speed
).show()    

Wind Summary Analysis:
+-----------------+-----------------+----------------+
|average_windspeed|highest_windspeed|lowest_windspeed|
+-----------------+-----------------+----------------+
|             2.52|              3.6|             1.1|
+-----------------+-----------------+----------------+



In [8]:
# Temperature by Hour

print("=== Average Temperature by Hour ===")
(
    df.groupBy(hour("timestamp").alias("hour"))                  # extract hour from timestamp
      .agg(round(avg("temperature"), 2).alias("avg_temp_C"))     # average temperature per hour
      .orderBy("hour")                                           # sort results by hour
      .show()
)

=== Average Temperature by Hour ===
+----+----------+
|hour|avg_temp_C|
+----+----------+
|   0|      21.5|
|   1|     22.43|
|   2|     23.88|
|   3|     25.18|
|   4|     26.28|
|   5|      26.9|
|  21|      20.7|
|  22|     20.83|
|  23|     20.98|
+----+----------+



In [9]:
# Temperature Standard deviation

print("=== Temperature Standard Deviation ===")
df.select(
    round(stddev("temperature"), 2).alias("temp_stddev_C")  # measures temperature variation
).show()

=== Temperature Standard Deviation ===
+-------------+
|temp_stddev_C|
+-------------+
|         2.25|
+-------------+



### Aggregation Conclusion

From the aggregations, temperatures between 20.7°C and 27.0°C were recorded, with an average of 23.2°C and a standard deviation of 2.25°C, indicating low variation during this period. Regarding wind speed, consistently low values were identified, ranging between 1.1 km/h and 3.6 km/h with an average of 2.52 km/h. The hourly analysis revealed a gradual increase throughout the early hours of the morning, peaking at 26.9°C at 05h.

> The Open-Meteo API updates every 15 minutes. Records were therefore collected at this interval to capture meaningful variation across the collection window. Collecting at shorter intervals would result in near-identical values, making the analysis less representative.

## Pattern Detection

### Pattern 1 — Heat Perception Gap

Temperature analysis revealed key patterns; however, it is important to understand whether temperatures tend to be equal, very similar, or show significant variation in relation to thermal sensation, as comfort and discomfort while cycling is more closely related to perceived temperature than to the actual recorded temperature.

In [10]:
# Heat Perception Gap — difference between apparent and actual temperature

df_heat_gap = df.withColumn(
    "heat_gap",                                                      # new column: gap between apparent and actual temperature
    round(col("apparent_temperature") - col("temperature"), 2)       # positive = hotter, negative = cooler
).withColumn(
    "perception_label",                                              # classify the gap
    when(col("heat_gap") >= 3, "Feels Much Hotter")                  # gap >= 3°C: high humidity effect
    .when(col("heat_gap") >= 1, "Feels Slightly Hotter")             # gap between 1–3°C: moderate effect
    .when(col("heat_gap") <= -1, "Feels Cooler")                     # negative gap: wind chill effect
    .otherwise("Neutral")                                            # gap close to 0: no significant difference
)

(df_heat_gap.groupBy("perception_label")
            .count()                                                 # count how many records fall into each label
            .orderBy("count", ascending=False)                       # order by count
            .show())                                                

+-----------------+-----+
| perception_label|count|
+-----------------+-----+
|Feels Much Hotter|   30|
+-----------------+-----+



> The fact that all collected records were classified as "Feels Much Hotter" highlights the importance of understanding that, despite temperature being a strong indicator, thermal sensation is even more relevant — as it tends to make cycling more uncomfortable than the actual temperature would suggest. It is worth noting that not everyone checks the apparent temperature, relying solely on the recorded value, which may lead to an underestimation of the real discomfort experienced while cycling.

### Pattern 2 — Wind Alert Flag

In addition to temperature, wind is also a relevant factor in bike rental behaviour. Therefore, it is important to understand how to evaluate wind intensity and create classifications that enrich the analysis. For this purpose, the Beaufort Scale will be used.

In [11]:
# Wind Alert — classify wind conditions based on Beaufort wind scale

df_wind_alert = df.withColumn(
    "wind_alert",                                                    # new column: wind speed classification
    when(
        col("wind_speed") >= 75,                                     # Beaufort 9+ (Strong Gale or above)
        "Dangerous"
    ).when(
        col("wind_speed") >= 50,                                     # Beaufort 7–8 (Gale)
        "High Wind Warning"
    ).when(
        col("wind_speed") >= 29,                                     # Beaufort 5–6 (Fresh to Strong Breeze)
        "Moderate Wind"
    ).otherwise("Normal")                                            # Beaufort 0–4: safe conditions
)

(df_wind_alert.groupBy("wind_alert")                                 # group records by wind alert label
              .count()                                               # count how many records fall into each label
              .orderBy("count", ascending=False)                     # order by count
              .show())

+----------+-----+
|wind_alert|count|
+----------+-----+
|    Normal|   30|
+----------+-----+



> The data collected indicates that during this period, wind speed was consistently classified as "Normal", not exceeding 29 km/h. This suggests that wind did not represent a discomfort factor for cyclists during the collection window. However, it is important to note that these records were collected within a single season, and results may vary significantly under different seasonal conditions.

## Final Conclusion
The real-time climate analysis of Seoul indicated consistently comfortable conditions for cycling during the collection period. All trends identified in the previous questions were confirmed and appropriately connected, reinforcing that while temperature is a key factor, thermal sensation proved to be even more decisive. It was also evidenced that wind speed, although moderate during the collection window, tends to present higher values in other seasons. Finally, these results demonstrate how real-time data processing is not only useful for weather monitoring, but also contributes to enriching the insights identified in Questions 1 and 2.

## Discussion: Batch Processing vs Real-Time Streaming

### The difference between batch processing and real-time streaming processing

**Batch processing** involves collecting and storing data in blocks, which accumulate into a large volume that is processed all at once — either as a single job or as a recurring task run at long intervals. A practical example of this is requesting a bank statement, where the result is only made available *"by the next business day"* rather than instantly, since batch processing happens at scheduled intervals: data is collected, stored, and then processed as one large batch rather than as it arrives.

**Real-time streaming processing**, on the other hand, handles data continuously and processes it the moment it is generated. This is typically necessary when dealing with large volumes of fast-moving data that must be analysed immediately or lose its usefulness. In stream processing, data flows continuously and is processed as each new piece of data arrives. A clear example of this is **fraud detection in banking systems**, where transactions must be flagged the instant they occur. Another example is **social media**, where huge volumes of data — messages, videos, comments — are generated every second, and must be processed instantly to generate personalised recommendations and advertisements for each user.

### The benefits of real-time analytics in data-driven systems

Real-time analytics offers many advantages. The fact that data is processed 
instantaneously means that decision-making becomes faster and more precise. 
As a consequence, anomalies tend to be detected earlier, preventing problems 
on a larger scale. There are also gains in terms of personalisation and 
scalability, given the dynamism that this data generates by being rapidly 
processed, which allows systems and companies to continuously adapt to growing 
demand. There are further advantages in the relationship between the company 
and the client, as the processed data generates increasingly personalised 
advertisements and recommendations, adapting very quickly to the behaviour 
of the client.

## References

- Open-Meteo.com (n.d.). Weather Forecast API | Open-Meteo.com. [online] Available at: https://open-meteo.com/en/docs [Accessed 30 Jun. 2026].
- Iqbal, M. (2026) Tutorial_8_realword_streaming. Lecture Notes. Dublin: CCT College Dublin.
- Snowplow (n.d.) Batch processing vs. stream processing. Available at: https://snowplow.io/blog/batch-processing-vs-stream-processing [Accessed: 30 June 2026].
- Wikipedia Contributors (2019). Beaufort scale. [online] Wikipedia. Available at: https://en.wikipedia.org/wiki/Beaufort_scale [Accessed: 1 July 2026].